# Practice Lab: Decision Trees — Entropy & Information Gain

*A self-contained recreation of the "Decision Trees" entropy/information-gain lab, built from scratch with an original dataset (mushroom edibility instead of the usual cat example) for practice purposes. No external `utils.py` or extra packages needed — just `numpy`, `pandas`, and `matplotlib`.*

In this notebook you will see how a decision tree decides *where* to split by comparing the **information gain** of different features.

$$\text{Information Gain} = H(p_1^{\text{node}}) - \left(w^{\text{left}}H\left(p_1^{\text{left}}\right) + w^{\text{right}}H\left(p_1^{\text{right}}\right)\right)$$

where $H$ is the **entropy**:

$$H(p_1) = -p_1 \log_2(p_1) - (1-p_1)\log_2(1-p_1)$$

$H$ is highest (=1) when $p_1 = 0.5$ (maximum uncertainty) and lowest (=0) when $p_1 = 0$ or $p_1 = 1$ (fully predictable / pure node).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

## 1. The entropy function

Let's define $H(p_1)$ and plot it to see how it behaves as $p_1$ varies from 0 to 1.

In [ ]:
def entropy(p):
    """Binary entropy H(p) in bits (log base 2)."""
    if p == 0 or p == 1:
        return 0
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p)

print("H(0.5) =", entropy(0.5))
print("H(0.1) =", entropy(0.1))
print("H(0.9) =", entropy(0.9))

In [ ]:
p_values = np.linspace(0.001, 0.999, 200)
h_values = [entropy(p) for p in p_values]

plt.figure(figsize=(6, 4))
plt.plot(p_values, h_values, color="teal", linewidth=2)
plt.axvline(0.5, color="gray", linestyle="--", linewidth=1)
plt.title("Entropy H(p) vs. probability of the positive class")
plt.xlabel("p (proportion of 'Edible' examples)")
plt.ylabel("Entropy H(p)")
plt.grid(alpha=0.3)
plt.show()

Notice entropy peaks at $p=0.5$ (a node that is a coin-flip between the two classes is the *least* predictable) and drops to 0 at the extremes (a pure node — every example belongs to one class).

## 2. The dataset: Mushroom Edibility 🍄

Instead of the usual "is it a cat" example, let's classify whether a mushroom is **edible** or **poisonous** from three simple, one-hot-encoded features:

| Mushroom | Cap Shape | Cap Color | Gill Size | Edible? |
|:---:|:---:|:---:|:---:|:---:|
| 🍄 #0 | Convex | Brown | Broad | ✅ Yes |
| 🍄 #1 | Flat | Brown | Broad | ✅ Yes |
| 🍄 #2 | Convex | Brown | Broad | ✅ Yes |
| 🍄 #3 | Flat | Brown | Broad | ✅ Yes |
| 🍄 #4 | Convex | White | Broad | ✅ Yes |
| 🍄 #5 | Flat | White | Narrow | ❌ No |
| 🍄 #6 | Convex | White | Narrow | ❌ No |
| 🍄 #7 | Flat | White | Narrow | ❌ No |
| 🍄 #8 | Convex | Brown | Narrow | ❌ No |
| 🍄 #9 | Flat | White | Narrow | ❌ No |

**One-hot encoding:**
- Cap Shape: Convex = 1, Flat = 0
- Cap Color: Brown = 1, White = 0
- Gill Size: Broad = 1, Narrow = 0

- `X_train`: each row has the 3 binary features above.
- `y_train`: 1 if the mushroom is edible, 0 if poisonous.

In [ ]:
X_train = np.array([
    [1, 1, 1],   # 0
    [0, 1, 1],   # 1
    [1, 1, 1],   # 2
    [0, 1, 1],   # 3
    [1, 0, 1],   # 4
    [0, 0, 0],   # 5
    [1, 0, 0],   # 6
    [0, 0, 0],   # 7
    [1, 1, 0],   # 8
    [0, 0, 0],   # 9
])

y_train = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])

feature_names = ["Cap Shape", "Cap Color", "Gill Size"]

pd.DataFrame(X_train, columns=feature_names).assign(Edible=y_train)

In [ ]:
# For instance, the first example
X_train[0]

This means mushroom #0 has a convex cap shape, a brown cap color, and broad gills — and it's edible.

## 3. Entropy of the root node

The root node contains every example. $p_1^{\text{node}}$ is the proportion of edible mushrooms:

$$p_1^{\text{node}} = \frac{5}{10} = 0.5$$

In [ ]:
p_root = sum(y_train) / len(y_train)
print("p_root =", p_root)
print("H(p_root) =", entropy(p_root))

## 4. Splitting on a feature

`split_indices` separates the examples into a **left** node (feature = 1) and a **right** node (feature = 0).

In [ ]:
def split_indices(X, index_feature):
    """
    Given a dataset and a feature index, return two lists of row indices:
    the left node has examples where that feature = 1, the right node
    has examples where that feature = 0.
      index_feature = 0 -> Cap Shape
      index_feature = 1 -> Cap Color
      index_feature = 2 -> Gill Size
    """
    left_indices = []
    right_indices = []
    for i, x in enumerate(X):
        if x[index_feature] == 1:
            left_indices.append(i)
        else:
            right_indices.append(i)
    return left_indices, right_indices

# Try splitting on Cap Shape (feature 0)
split_indices(X_train, 0)

## 5. Weighted entropy of a split

For a split we need:
- $w^{\text{left}}$, $w^{\text{right}}$ — the *proportion of examples* in each branch.
- $p^{\text{left}}$, $p^{\text{right}}$ — the *proportion of edible mushrooms* in each branch.

In [ ]:
def weighted_entropy(X, y, left_indices, right_indices):
    """Takes the split indices and returns the weighted entropy of the two child nodes."""
    w_left = len(left_indices) / len(X)
    w_right = len(right_indices) / len(X)
    p_left = sum(y[left_indices]) / len(left_indices)
    p_right = sum(y[right_indices]) / len(right_indices)

    weighted_entropy = w_left * entropy(p_left) + w_right * entropy(p_right)
    return weighted_entropy

left_indices, right_indices = split_indices(X_train, 0)
weighted_entropy(X_train, y_train, left_indices, right_indices)

## 6. Information gain

$$\text{Information Gain} = H(p_1^{\text{node}}) - \text{weighted entropy of the split}$$

In [ ]:
def information_gain(X, y, left_indices, right_indices):
    """X, y are the examples/labels *in the node being split*."""
    p_node = sum(y) / len(y)
    h_node = entropy(p_node)
    w_entropy = weighted_entropy(X, y, left_indices, right_indices)
    return h_node - w_entropy

information_gain(X_train, y_train, left_indices, right_indices)

So splitting on **Cap Shape** alone gains us very little information. Let's check all three features.

In [ ]:
for i, feature_name in enumerate(feature_names):
    left_indices, right_indices = split_indices(X_train, i)
    i_gain = information_gain(X_train, y_train, left_indices, right_indices)
    print(f"Feature: {feature_name:10s} | information gain if we split the root node on it: {i_gain:.4f}")

**Gill Size** wins by a landslide — an information gain of **1.0**, the maximum possible! That means splitting on Gill Size alone perfectly separates edible from poisonous mushrooms in this dataset. Cap Color is a decent, moderate predictor, and Cap Shape barely helps at all.

## 7. Building the tree recursively

A real decision tree repeats this process on every child node until it hits a stopping condition:
- the tree depth exceeds a maximum,
- a node is already **pure** (only one class present), or
- the best available information gain is (near) zero.

Below is a small, self-contained recursive tree builder — no extra libraries required.

In [ ]:
def compute_entropy(y):
    if len(y) == 0:
        return 0
    p1 = np.sum(y) / len(y)
    return entropy(p1)

def get_best_split(X, y):
    """Returns (best_feature_index, best_information_gain) for this node."""
    best_feature, best_ig = -1, -1
    for i in range(X.shape[1]):
        left, right = split_indices(X, i)
        if len(left) == 0 or len(right) == 0:
            continue
        ig = information_gain(X, y, left, right)
        if ig > best_ig:
            best_ig, best_feature = ig, i
    return best_feature, best_ig

def build_tree(X, y, indices, feature_names, max_depth, current_depth=0):
    """Recursively builds a tree (as nested dicts) and returns the root node."""
    y_node = y[indices]
    node = {
        "indices": indices,
        "depth": current_depth,
        "entropy": compute_entropy(y_node),
        "n_samples": len(indices),
        "n_edible": int(np.sum(y_node)),
    }

    # Stopping criteria
    if current_depth >= max_depth or node["entropy"] == 0 or len(indices) < 2:
        node["leaf"] = True
        return node

    X_node = X[indices]
    best_feature, best_ig = get_best_split(X_node, y_node)
    if best_feature == -1 or best_ig <= 1e-9:
        node["leaf"] = True
        return node

    left_local, right_local = split_indices(X_node, best_feature)
    left_indices = [indices[i] for i in left_local]
    right_indices = [indices[i] for i in right_local]

    node["leaf"] = False
    node["feature"] = feature_names[best_feature]
    node["ig"] = best_ig
    node["left"] = build_tree(X, y, left_indices, feature_names, max_depth, current_depth + 1)
    node["right"] = build_tree(X, y, right_indices, feature_names, max_depth, current_depth + 1)
    return node

def print_tree(node, prefix="", branch=""):
    tag = f"[{node['n_edible']}/{node['n_samples']} edible, entropy={node['entropy']:.2f}]"
    if node["leaf"]:
        print(f"{prefix}{branch}Leaf {tag}")
        return
    print(f"{prefix}{branch}{node['feature']}? (IG={node['ig']:.2f}) {tag}")
    print_tree(node["left"], prefix + "    ", "Yes -> ")
    print_tree(node["right"], prefix + "    ", "No  -> ")

In [ ]:
root_indices = list(range(len(X_train)))

print("=== Tree with max_depth = 1 ===")
tree_depth1 = build_tree(X_train, y_train, root_indices, feature_names, max_depth=1)
print_tree(tree_depth1)

A single split on **Gill Size** already gives two perfectly pure leaves (0 entropy each) — every "Broad" gill mushroom is edible, every "Narrow" gill mushroom is poisonous.

## 8. What if we allow more depth?

In [ ]:
print("=== Tree with max_depth = 2 ===")
tree_depth2 = build_tree(X_train, y_train, root_indices, feature_names, max_depth=2)
print_tree(tree_depth2)

Notice the depth-2 tree is **identical** to the depth-1 tree. That's the second stopping criterion in action: once a node is already pure (entropy = 0), there's nothing left to gain by splitting it further, so the recursion stops early even though `max_depth` would allow more levels.

## 9. Visualizing the tree

Finally, let's draw both trees side by side with `matplotlib` (no `graphviz` install needed).

In [ ]:
def assign_positions(node, depth, x_counter, positions):
    """Assigns an (x, y) plotting position to every node via in-order traversal."""
    if node["leaf"]:
        x = x_counter[0]
        x_counter[0] += 1
        positions[id(node)] = (x, -depth)
        return x
    x_left = assign_positions(node["left"], depth + 1, x_counter, positions)
    x_right = assign_positions(node["right"], depth + 1, x_counter, positions)
    x = (x_left + x_right) / 2
    positions[id(node)] = (x, -depth)
    return x

def draw_tree(node, ax, positions):
    x, y = positions[id(node)]
    if node["leaf"]:
        if node["n_edible"] == node["n_samples"]:
            label, color = "Edible", "#d5f5d0"
        elif node["n_edible"] == 0:
            label, color = "Poisonous", "#f5d0d0"
        else:
            label, color = "Mixed", "#f5eecb"
        text = f"{label}\n{node['n_edible']}/{node['n_samples']}"
    else:
        text = f"{node['feature']}?\nIG={node['ig']:.2f}\n{node['n_edible']}/{node['n_samples']}"
        color = "#dbe9f7"
    ax.text(x, y, text, ha="center", va="center", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.4", fc=color, ec="black"))
    if not node["leaf"]:
        for child, lbl in [(node["left"], "Yes (=1)"), (node["right"], "No (=0)")]:
            cx, cy = positions[id(child)]
            ax.plot([x, cx], [y - 0.05, cy + 0.05], color="gray", zorder=0)
            ax.text((x + cx) / 2, (y + cy) / 2, lbl, fontsize=8, color="dimgray",
                     ha="center", va="center", backgroundcolor="white")
            draw_tree(child, ax, positions)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, tree, title in [(axes[0], tree_depth1, "max_depth = 1"),
                         (axes[1], tree_depth2, "max_depth = 2")]:
    positions = {}
    assign_positions(tree, 0, [0], positions)
    draw_tree(tree, ax, positions)
    xs = [p[0] for p in positions.values()]
    ys = [p[1] for p in positions.values()]
    ax.set_xlim(min(xs) - 0.6, max(xs) + 0.6)
    ax.set_ylim(min(ys) - 0.4, max(ys) + 0.4)
    ax.axis("off")
    ax.set_title(title)

plt.tight_layout()
plt.show()

## 10. Try it yourself

Some ideas to extend this notebook:
- Add a 4th feature (e.g. "Spore Print Color") and see how it changes the information gain ranking.
- Add more, noisier examples where no single feature gives a perfect split, and watch the tree need depth 2 or 3 to fully separate the classes.
- Swap in a real dataset (e.g. the classic [UCI Mushroom dataset](https://archive.ics.uci.edu/dataset/73/mushroom)) and run the same `build_tree` function on it.

Congratulations — you've rebuilt the entropy / information-gain / decision-tree pipeline from scratch! 🎉